# Rumore e ritorno: come funziona la diffusione

Il codice del capitolo [«Rumore e ritorno: come funziona la diffusione»](https://book.paithon.it/main/ModelliDiffusione/come-funziona.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision

## Rumore e ritorno: come funziona la diffusione

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/come-funziona.html)


### La diffusione in miniatura: una spirale di punti


In [ ]:
import numpy as npimport torchfrom torch import nntorch.manual_seed(0)rng = np.random.default_rng(0)# --- Dati: 2000 punti disposti a spirale, coordinate in [-1, 1] ---n = 2000angolo = 3.0 * np.pi * np.sqrt(rng.uniform(size=n))    # angolo lungo la spiraleraggio = angolo / (3.0 * np.pi)                        # il raggio cresce con l'angolospirale = np.stack([raggio * np.cos(angolo),                    raggio * np.sin(angolo)], axis=1)  # shape (2000, 2)spirale += 0.02 * rng.standard_normal(spirale.shape)   # leggero spessore del trattox0 = torch.tensor(spirale, dtype=torch.float32)        # (2000, 2)# --- Schedule del rumore: lo stesso di DDPM ---T = 1000beta = torch.linspace(1e-4, 0.02, T)       # beta_t, shape (T,)alpha = 1.0 - beta                         # alpha_talpha_bar = torch.cumprod(alpha, dim=0)    # alpha_t barrato, shape (T,)def rumorizza(x0, t, eps):    """Forma chiusa dell'andata: x_t dato x_0, per t interi in [0, T-1]."""    ab = alpha_bar[t].unsqueeze(1)                     # (B, 1)    return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps    # (B, 2)

### --- Schedule del rumore: lo stesso di DDPM ---


In [ ]:
def embedding_tempo(t, dim=16):    """Embedding sinusoidale del passo t: da (B,) a (B, dim)."""    freq = torch.exp(torch.arange(dim // 2) * (-np.log(10000.0) / (dim // 2)))    ang = t.float().unsqueeze(1) * freq.unsqueeze(0)   # (B, dim/2)    return torch.cat([ang.sin(), ang.cos()], dim=1)    # (B, dim)class PredittoreRumore(nn.Module):    """La rete epsilon_theta(x_t, t): un MLP al posto della U-Net."""    def __init__(self, dim_t=16, dim_h=128):        super().__init__()        self.dim_t = dim_t        self.rete = nn.Sequential(            nn.Linear(2 + dim_t, dim_h), nn.SiLU(),            nn.Linear(dim_h, dim_h), nn.SiLU(),            nn.Linear(dim_h, 2),                       # stima del rumore 2D        )    def forward(self, x, t):        emb = embedding_tempo(t, self.dim_t)           # (B, dim_t)        return self.rete(torch.cat([x, emb], dim=1))   # (B, 2)

In [ ]:
modello = PredittoreRumore()ottimizzatore = torch.optim.Adam(modello.parameters(), lr=2e-3)for passo in range(4000):    idx = torch.randint(0, n, (256,))         # minibatch di 256 punti    batch = x0[idx]                           # (256, 2)    t = torch.randint(0, T, (256,))           # un livello di rumore per esempio    eps = torch.randn_like(batch)             # il rumore "vero" (la soluzione)    x_t = rumorizza(batch, t, eps)            # (256, 2)    predetto = modello(x_t, t)                # (256, 2), rumore stimato    loss = ((eps - predetto) ** 2).mean()     # MSE: la loss semplice di DDPM    ottimizzatore.zero_grad()    loss.backward()    ottimizzatore.step()    if passo % 1000 == 0:        print(f"passo {passo:4d}  loss {loss.item():.3f}")

In [ ]:
@torch.no_grad()def campiona(n_campioni=1000):    """Percorre la catena inversa da x_T (rumore puro) a x_0."""    x = torch.randn(n_campioni, 2)                        # x_T ~ N(0, I)    for t in reversed(range(T)):        t_batch = torch.full((n_campioni,), t)            # (B,), tutti uguali a t        eps_pred = modello(x, t_batch)                    # rumore stimato        coeff = beta[t] / (1.0 - alpha_bar[t]).sqrt()        media = (x - coeff * eps_pred) / alpha[t].sqrt()  # mu_theta(x_t, t)        if t > 0:            x = media + beta[t].sqrt() * torch.randn_like(x)  # sigma_t * z        else:            x = media                    # ultimo passo: niente rumore fresco    return x                             # (n_campioni, 2)nuovi = campiona()print(nuovi.shape)   # torch.Size([1000, 2]): punti nuovi, disposti a spirale

## Lo spazio latente: Stable Diffusion

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/stable-diffusion.html)


### Dieci righe di Python


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
from diffusers import StableDiffusionPipeline

# carica l'intera pipeline (VAE + U-Net + CLIP + campionatore)
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,  # mezza precisione: meno memoria
)
pipe = pipe.to("cuda")          # sposta tutto sulla GPU

immagine = pipe(
    prompt="a black cat jumping on a wall, watercolor",
    negative_prompt="blurry, deformed, watermark",
    guidance_scale=7.5,         # il peso w della guidance
    num_inference_steps=50,     # i passi di denoising nel latente
).images[0]

immagine.save("gatto_acquerello.png")
```


## Quando la diffusione incontra i Transformer: DiT

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/diffusion-transformer.html)


### Un DiT in miniatura


In [ ]:
import mathimport torchfrom torch import nntorch.manual_seed(0)def embedding_tempo(t, dim=128):    """Embedding sinusoidale del passo t: da (B,) a (B, dim)."""    freq = torch.exp(-math.log(10000.0) * torch.arange(dim // 2) / (dim // 2))    ang = t.float().unsqueeze(1) * freq.unsqueeze(0)     # (B, dim/2)    return torch.cat([ang.sin(), ang.cos()], dim=1)      # (B, dim)class BloccoDiT(nn.Module):    """Attenzione + MLP, con modulazione adaLN-zero dal condizionamento."""    def __init__(self, d=128, teste=4):        super().__init__()        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)  # LN "nuda"        self.attn = nn.MultiheadAttention(d, teste, batch_first=True)        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(),                                 nn.Linear(4 * d, d))        # dal condizionamento: shift, scale e gate per i due sotto-strati        self.manopole = nn.Linear(d, 6 * d)        nn.init.zeros_(self.manopole.weight)   # adaLN-ZERO: blocco = identita'        nn.init.zeros_(self.manopole.bias)    def forward(self, x, c):        # x: (B, T, d) token del latente; c: (B, d) tempo + classe        b1, g1, a1, b2, g2, a2 = self.manopole(c).chunk(6, dim=1)  # 6 x (B, d)        h = self.norm1(x) * (1 + g1.unsqueeze(1)) + b1.unsqueeze(1)        att, _ = self.attn(h, h, h, need_weights=False)        x = x + a1.unsqueeze(1) * att            # gate a1: vale 0 all'inizio        h = self.norm2(x) * (1 + g2.unsqueeze(1)) + b2.unsqueeze(1)        x = x + a2.unsqueeze(1) * self.mlp(h)    # gate a2, idem        return xclass MiniDiT(nn.Module):    """DiT minimale: patchify, blocchi Transformer, patch di rumore in uscita."""    def __init__(self, canali=4, lato=32, patch=2, d=128, blocchi=4, classi=10):        super().__init__()        self.canali, self.lato, self.patch = canali, lato, patch        n_token = (lato // patch) ** 2                       # (32/2)^2 = 256        self.patchify = nn.Conv2d(canali, d, kernel_size=patch, stride=patch)        self.pos = nn.Parameter(torch.zeros(1, n_token, d))  # posizioni apprese        self.emb_classe = nn.Embedding(classi, d)        self.blocchi = nn.ModuleList([BloccoDiT(d) for _ in range(blocchi)])        self.finale = nn.Linear(d, patch * patch * canali)   # token -> sua patch    def forward(self, z, t, y):        # z: (B, 4, 32, 32) latente rumoroso; t: (B,) passo; y: (B,) classe        x = self.patchify(z)                     # (B, d, 16, 16)        x = x.flatten(2).transpose(1, 2)         # (B, 256, d): i token        x = x + self.pos        c = embedding_tempo(t, x.shape[-1]) + self.emb_classe(y)   # (B, d)        for blocco in self.blocchi:            x = blocco(x, c)        x = self.finale(x)                       # (B, 256, patch*patch*4)        # ricompone le patch: l'uscita ha la stessa forma dell'ingresso        B, g, p, C = z.shape[0], self.lato // self.patch, self.patch, self.canali        x = x.view(B, g, g, p, p, C)             # (B, 16, 16, 2, 2, 4)        x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, self.lato, self.lato)        return x                                 # (B, 4, 32, 32): rumore stimatomodello = MiniDiT()z = torch.randn(2, 4, 32, 32)      # due latenti fittizi, come quelli del VAEt = torch.randint(0, 1000, (2,))   # un passo di rumore per ciascunoy = torch.randint(0, 10, (2,))     # una classe per ciascunoprint(modello(z, t, y).shape)      # torch.Size([2, 4, 32, 32])print(sum(p.numel() for p in modello.parameters()))  # 1225616: ~1.2 milioni